<a href="https://github.com/N3iKos/segsmaker-fast">
  <img alt="GitHub repo" src="https://img.shields.io/badge/GitHub-6e5494?style=for-the-badge&logo=github&logoColor=white"/>
</a><br>

*   Get your Civitai API Key from [here](https://civitai.com/user/account)
*   Get your Huggingface token from [here](https://huggingface.co/settings/tokens)

In [ ]:
# @title <b><font color='orange'>WebUI Installer</font></b> {"display-mode":"form"}

#@markdown ### 🚀 Step 1 - Choose WebUI
Webui = 'Forge-Neo' # @param ["A1111", "Forge", "ReForge", "ReForge-old", "Forge-Classic", "Forge-Neo", "ComfyUI", "SwarmUI"]

#@markdown ### 🔑 Step 2 - API Keys
Civitai___Key = '' # @param { type: "string", placeholder: "Your Civitai API Key (required)" }
HF_Read_Token = '' # @param { type: "string", placeholder: "Your Huggingface READ Token (optional)" }

#@markdown ### 📂 Step 3 - Google Drive
Mount__GDrive = 'No' # @param ["Yes", "No"]

mount = Mount__GDrive

if mount == 'Yes':
    from google.colab import drive
    drive.mount('/content/drive')

!curl -sLo /content/setup.py https://github.com/N3iKos/segsmaker-fast/raw/main/script/KC/setup.py
%run /content/setup.py --webui="$Webui" --civitai_key="$Civitai___Key" --hf_read_token="$HF_Read_Token"

if mount == 'Yes':
    from pathlib import Path

    d = Path('/content/drive/MyDrive/Segsmaker')

    for n, p in {'checkpoint': CKPT, 'lora': LORA, 'vae': VAE, 'embeddings': Embeddings}.items():
        f = d / n
        f.mkdir(parents=True, exist_ok=True)
        s = p / f'drive-{n}'
        if not s.exists():
            s.symlink_to(f, target_is_directory=True)

    !rm -rf $WebUI_Output
    o = d / {'ComfyUI': 'comfyui-output', 'SwarmUI': 'swarmui-output'}.get(Webui, 'output')
    o.mkdir(parents=True, exist_ok=True)
    if not WebUI_Output.exists():
        WebUI_Output.symlink_to(o, target_is_directory=True)

    if Webui not in {'ComfyUI', 'SwarmUI'}:
        wc = WebUI / 'cache'
        !rm -rf $wc
        c = d / 'cache'
        c.mkdir(parents=True, exist_ok=True)
        if not wc.exists():
            wc.symlink_to(c, target_is_directory=True)

In [ ]:
# @title <b><font color='orange'>Model Downloader - 5 Checkpoint + 5 LoRA + VAE</font></b> {"display-mode":"form"}

#@markdown ### 📥 Checkpoints (Models)
Checkpoint_1 = "" #@param {type:"string"}
Checkpoint_2 = "" #@param {type:"string"}
Checkpoint_3 = "" #@param {type:"string"}
Checkpoint_4 = "" #@param {type:"string"}
Checkpoint_5 = "" #@param {type:"string"}

#@markdown ### 📥 LoRAs
Lora_1 = "" #@param {type:"string"}
Lora_2 = "" #@param {type:"string"}
Lora_3 = "" #@param {type:"string"}
Lora_4 = "" #@param {type:"string"}
Lora_5 = "" #@param {type:"string"}

#@markdown ### 📥 VAE
VAE_URL = "" #@param {type:"string", placeholder:"URL or leave empty"}

#@markdown ### ⚡ Speed Options
Parallel_Download = True #@param {type:"boolean"}
Max_Workers = 3 #@param {type:"slider", min:1, max:5, step:1}

#@markdown ### 💾 Google Drive Integration
Load_From_Drive = False #@param {type:"boolean"}

import os
from pathlib import Path
from nenen88 import download_parallel, netorare_safe

gdrive_base = Path('/content/drive/MyDrive/Segsmaker')
if Load_From_Drive and gdrive_base.exists():
    ckpt_dir = gdrive_base / 'checkpoint'
    lora_dir = gdrive_base / 'lora'
    vae_dir = gdrive_base / 'vae'
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    lora_dir.mkdir(parents=True, exist_ok=True)
    vae_dir.mkdir(parents=True, exist_ok=True)
    print("💾 Saving downloaded models/loras directly to Google Drive.")
else:
    if Load_From_Drive:
        print("⚠️ Google Drive not mounted or Segsmaker folder not found. Falling back to local temporary storage.")
    ckpt_dir = CKPT
    lora_dir = LORA
    vae_dir = VAE

downloads = []
for cp in [Checkpoint_1, Checkpoint_2, Checkpoint_3, Checkpoint_4, Checkpoint_5]:
    if cp.strip(): downloads.append((cp.strip(), ckpt_dir))
for lr in [Lora_1, Lora_2, Lora_3, Lora_4, Lora_5]:
    if lr.strip(): downloads.append((lr.strip(), lora_dir))
if VAE_URL.strip():
    downloads.append((VAE_URL.strip(), vae_dir))

if downloads:
    if Parallel_Download:
        print(f"🚀 Downloading {len(downloads)} items in parallel using {Max_Workers} workers...")
        download_parallel(downloads, max_workers=Max_Workers)
    else:
        print(f"📥 Downloading {len(downloads)} items sequentially...")
        for line, dest in downloads:
            netorare_safe(line, dest)
    print("\n✅ All downloads completed!")
else:
    print("ℹ️ No URLs provided for download.")

In [ ]:
# @title <b><font color='orange'>Extra Assets - Extensions, Embeddings, Upscalers</font></b> {"display-mode":"form"}

#@markdown ### 🔌 Extensions / ComfyUI Custom Nodes
Extension_1 = "" #@param {type:"string", placeholder:"git clone URL or leave empty"}
Extension_2 = "" #@param {type:"string", placeholder:"git clone URL or leave empty"}
Extension_3 = "" #@param {type:"string", placeholder:"git clone URL or leave empty"}
Extension_4 = "" #@param {type:"string", placeholder:"git clone URL or leave empty"}
Extension_5 = "" #@param {type:"string", placeholder:"git clone URL or leave empty"}

#@markdown ### 📂 Embeddings
Embedding_1 = "" #@param {type:"string"}
Embedding_2 = "" #@param {type:"string"}
Embedding_3 = "" #@param {type:"string"}

#@markdown ### ⬆️ Upscalers
Upscaler_1 = "" #@param {type:"string"}
Upscaler_2 = "" #@param {type:"string"}
Upscaler_3 = "" #@param {type:"string"}

#@markdown ### ⚡ Speed Options
Assets_Parallel_Download = True #@param {type:"boolean"}
Assets_Max_Workers = 3 #@param {type:"slider", min:1, max:5, step:1}

import os
from pathlib import Path
from nenen88 import download_parallel, netorare_safe

extensions_dir = Extensions
embeddings_dir = Embeddings
upscalers_dir = Upscalers

def git_clone_safe(git_url, target_dir):
    import subprocess
    if not git_url.strip(): return
    url = git_url.strip()
    if url.startswith("git clone "):
        url = url[len("git clone "):].strip()
    repo_name = url.split("/")[-1].replace(".git", "")
    dest_path = Path(target_dir) / repo_name
    if dest_path.exists():
        print(f"🔌 Extension {repo_name} already exists. Skipping clone.")
        return
    print(f"🔌 Cloning {url}...")
    try:
        subprocess.run(["git", "clone", url], cwd=str(target_dir), stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        print(f"🔌 Extension {repo_name} cloned successfully!")
    except Exception as e:
        print(f"❌ Error cloning {url}: {e}")

ext_list = [Extension_1, Extension_2, Extension_3, Extension_4, Extension_5]
ext_list = [e.strip() for e in ext_list if e.strip()]

if ext_list:
    if Assets_Parallel_Download:
        import concurrent.futures
        with concurrent.futures.ThreadPoolExecutor(max_workers=Assets_Max_Workers) as executor:
            futures = [executor.submit(git_clone_safe, ext, extensions_dir) for ext in ext_list]
            concurrent.futures.wait(futures)
    else:
        for ext in ext_list:
            git_clone_safe(ext, extensions_dir)

downloads = []
for emb in [Embedding_1, Embedding_2, Embedding_3]:
    if emb.strip(): downloads.append((emb.strip(), embeddings_dir))
for ups in [Upscaler_1, Upscaler_2, Upscaler_3]:
    if ups.strip(): downloads.append((ups.strip(), upscalers_dir))

if downloads:
    if Assets_Parallel_Download:
        print(f"🚀 Downloading {len(downloads)} assets in parallel using {Assets_Max_Workers} workers...")
        download_parallel(downloads, max_workers=Assets_Max_Workers)
    else:
        print(f"📥 Downloading {len(downloads)} assets sequentially...")
        for line, dest in downloads:
            netorare_safe(line, dest)
    print("\n✅ Extra assets downloaded!")
else:
    if not ext_list:
        print("ℹ️ No assets or extensions provided for download.")

In [ ]:
# @title <b><font color='orange'>FLUX Model Downloader</font></b> {"display-mode":"form"}

#@markdown ### 🌀 Choose FLUX Variant
FLUX_Variant = "FLUX.1-schnell (Fast, 4-step)" # @param ["FLUX.1-schnell (Fast, 4-step)", "FLUX.1-dev (Quality, 20-step)"]

#@markdown ### 🔗 Component URLs (Leave blank to use default FP8 recompiled models)
FLUX_Unet = "" #@param {type:"string"}
FLUX_Clip_L = "" #@param {type:"string"}
FLUX_T5XXL = "" #@param {type:"string"}
FLUX_VAE = "" #@param {type:"string"}

#@markdown ### ⚡ Speed Options
Parallel_FLUX_Download = True #@param {type:"boolean"}
FLUX_Max_Workers = 2 #@param {type:"slider", min:1, max:4, step:1}

from nenen88 import download_parallel, netorare_safe
from pathlib import Path

schnell_unet = "https://huggingface.co/Comfy-Org/flux1-schnell/resolve/main/flux1-schnell-fp8.safetensors"
dev_unet = "https://huggingface.co/Comfy-Org/flux1-dev/resolve/main/flux1-dev-fp8.safetensors"
default_clip_l = "https://huggingface.co/comfyanonymous/personal_sharing/resolve/main/clip_l.safetensors"
default_t5xxl = "https://huggingface.co/comfyanonymous/personal_sharing/resolve/main/t5xxl_fp8_e4m3fn.safetensors"
default_vae = "https://huggingface.co/black-forest-labs/FLUX.1-schnell/resolve/main/ae.safetensors"

unet_url = FLUX_Unet.strip() if FLUX_Unet.strip() else (schnell_unet if "schnell" in FLUX_Variant.lower() else dev_unet)
clip_l_url = FLUX_Clip_L.strip() if FLUX_Clip_L.strip() else default_clip_l
t5xxl_url = FLUX_T5XXL.strip() if FLUX_T5XXL.strip() else default_t5xxl
vae_url = FLUX_VAE.strip() if FLUX_VAE.strip() else default_vae

unet_dir = UNET
clip_dir = CLIP
vae_dir = VAE

downloads = [
    (unet_url, unet_dir),
    (clip_l_url, clip_dir),
    (t5xxl_url, clip_dir),
    (vae_url, vae_dir)
]

print(f"🌀 Preparing FLUX components for variant: {FLUX_Variant}")
if Parallel_FLUX_Download:
    print(f"🚀 Downloading FLUX components in parallel using {FLUX_Max_Workers} workers...")
    download_parallel(downloads, max_workers=FLUX_Max_Workers)
else:
    print("📥 Downloading FLUX components sequentially...")
    for line, dest in downloads:
        netorare_safe(line, dest)
print("\n✅ FLUX models downloaded and ready!")

In [ ]:
''' Controlnet '''
%run $Controlnet_Widget

In [ ]:
# @title <b><font color='orange'>Launcher WebUI</font></b> {"display-mode":"form"}

#@markdown ### ℹ️ Instruction: *Select the same WebUI that you installed in the first cell.*
Software = 'Forge-Neo' # @param ["A1111", "Forge", "ReForge", "ReForge-old", "Forge-Classic", "Forge-Neo", "ComfyUI", "SwarmUI"]

#@markdown ### 🔑 Tunnel Tokens (Optional)
Ngrok_Token = "" #@param {type:"string"}
Zrok_Token = "" #@param {type:"string"}

#@markdown ### ⚙️ Additional Settings
Extra_Args = "--xformers" #@param {type:"string"}
Skip_ComfyUI_Check = False #@param {type:"boolean"}
Skip_Widget = False #@param {type:"boolean"}

import os
from pathlib import Path

webui_dir = Path.home() / Software
if not webui_dir.exists():
    try:
        webui_dir = WebUI
    except NameError:
        webui_dir = Path.home() / 'stable-diffusion-webui'

if webui_dir.exists():
    os.chdir(str(webui_dir))
    print(f"🚀 Launching {Software} in {webui_dir}...")
else:
    print(f"❌ Error: WebUI directory {webui_dir} not found. Please install it in the first cell.")

args = []
if Skip_ComfyUI_Check:
    args.append("--skip-comfyui-check")
if Ngrok_Token.strip():
    args.append(f"--N={Ngrok_Token.strip()}")
if Zrok_Token.strip():
    args.append(f"--Z={Zrok_Token.strip()}")
if Extra_Args.strip():
    args.append(Extra_Args.strip())

cmd = f"segsmaker.py {' '.join(args)}"
%run $cmd

## 🛠️ Extras & Hidden Features
Gunakan Cell di bawah ini untuk pendaftaran akun tunnel, memeriksa sisa penyimpanan, atau membuat file zip output.

In [ ]:
''' Register ZROK Account '''
%zrok_register

In [ ]:
''' Check Storage Usage '''
%storage

In [ ]:
%%zipping

name    = 'outputs_archive'
inputs  = $WebUI_Output
outputs = $HOME